# Derivation: The EWLS Recursion and the Decaying Prior
In this derivation, we obtain the exponentially weighted least squares (EWLS) recursion that the L7b lecture uses to track the single index model (SIM) parameters $(\alpha_{i}, \beta_{i}, \sigma_{\varepsilon,i})$ of asset $i$ as new trading days arrive. We derive the running sufficient statistics $(\mathbf{A}_{t}, \mathbf{b}_{t}, c_{t})$ from the weighted normal equations, show that they update by one decayed-plus-rank-one step per day, recover the parameter estimate and the residual scale from them, and then make the prior precise: seeding the moments with $N_{0}$ pseudo-observations returns the prior exactly, and the seeded recursion minimizes the data loss plus a prior-centered quadratic whose weight decays like the data.

> __Learning Objectives:__
>
> By the end of this derivation, you should be able to:
> * __Derive the weighted normal equations:__ Write the exponentially weighted squared-residual loss with a decay factor set by the half-life, set its gradient to zero, and read off why a two-by-two matrix, a two-vector, and a scalar are sufficient for everything the estimator reports.
> * __Show the recursion and the closed forms:__ Show that each running moment updates as one decayed-plus-rank-one step when time advances, solve the two-by-two system for the intercept and beta, and recover the residual scale from the same three moments through a cancellation at the optimum.
> * __State what the prior does:__ Seed the moments so that a stated number of pseudo-observations carries a calibrated prior, prove that the seeded state returns the prior exactly, and prove that the seeded recursion minimizes the data loss plus a decaying prior-centered quadratic, so that the half-life and the prior weight are the only two knobs.

Let's get started!
___

## The SIM in Vector Form and the Weighted Loss
We want to track the SIM parameters $(\alpha_{i}, \beta_{i})$ of asset $i$ as new trading days arrive, without storing every past observation. Fix asset $i$ and, on each trading day $s = 1, 2, \ldots, t$, observe the asset's daily growth rate $g_{i,s}$ and the market's daily growth rate $g_{M,s}$, both in annualized units (inverse years, L6a). Stack the regressor with an intercept into the vector $\mathbf{x}_{s} = (1,\; g_{M,s})^{\top}\in\mathbb{R}^{2}$, write the parameter vector as $\boldsymbol{\theta}_{i} = (\alpha_{i},\;\beta_{i})^{\top}$, and write the response as $y_{s} = g_{i,s}$, so that the SIM regression of L6a on day $s$ reads:
$$
y_{s} = \mathbf{x}_{s}^{\top}\boldsymbol{\theta}_{i} + \varepsilon_{i,s}
$$
where $\varepsilon_{i,s}$ is the residual (inverse years). Old observations should fade as new ones arrive. Pick a __half-life__ $t_{1/2} > 0$ in trading days and define the __decay factor__ $\delta = 2^{-1/t_{1/2}}\in(0,1)$; the weight at time $t$ on the observation made on day $s\le t$ is $\delta^{\,t-s}$, so an observation $t_{1/2}$ days old carries half the weight of today's, because $\delta^{t_{1/2}} = 1/2$. The EWLS estimate of $\boldsymbol{\theta}_{i}$ at time $t$ is the minimizer of the weighted squared-residual loss:
$$
L_{t}(\boldsymbol{\theta}) = \sum_{s=1}^{t}\delta^{\,t-s}\left(y_{s} - \mathbf{x}_{s}^{\top}\boldsymbol{\theta}\right)^{2}
$$
This is a quadratic in $\boldsymbol{\theta}$, so the minimizer is unique whenever the weighted regressor moments have full rank, which holds as soon as the market growth rates in the weighted window are not all equal. Ordinary least squares (L6a) is the case $\delta = 1$: every observation carries weight one and nothing is forgotten.
___

## Weighted Normal Equations and Sufficient Statistics
Setting $\nabla_{\boldsymbol{\theta}}L_{t} = \mathbf{0}$ gives $-2\sum_{s\le t}\delta^{t-s}\mathbf{x}_{s}(y_{s} - \mathbf{x}_{s}^{\top}\boldsymbol{\theta}) = \mathbf{0}$, which rearranges to the weighted normal equations $\mathbf{A}_{t}\boldsymbol{\theta} = \mathbf{b}_{t}$ with the two weighted moments the optimum needs, and one extra weighted scalar moment $c_{t}$ that returns when we estimate the residual scale, defined by:
$$
\mathbf{A}_{t} = \sum_{s=1}^{t}\delta^{\,t-s}\,\mathbf{x}_{s}\mathbf{x}_{s}^{\top}\in\mathbb{R}^{2\times 2},\qquad
\mathbf{b}_{t} = \sum_{s=1}^{t}\delta^{\,t-s}\,\mathbf{x}_{s}\,y_{s}\in\mathbb{R}^{2},\qquad
c_{t} = \sum_{s=1}^{t}\delta^{\,t-s}\,y_{s}^{2}\in\mathbb{R}
$$
Everything EWLS needs is encoded in these three running quantities. Because $\mathbf{x}_{s}\mathbf{x}_{s}^{\top}$ has a one in its top-left entry, $[\mathbf{A}_{t}]_{11} = \sum_{s\le t}\delta^{t-s}$ is the total weight, a count of current-observation equivalents that tends to $1/(1-\delta)$ (the variance-based effective sample size $(\sum_{s}w_{s})^{2}/\sum_{s}w_{s}^{2}$ tends to $(1+\delta)/(1-\delta)$, about twice that); the off-diagonal entry is $\sum_{s}\delta^{t-s}g_{M,s}$ (inverse years) and $[\mathbf{A}_{t}]_{22} = \sum_{s}\delta^{t-s}g_{M,s}^{2}$ (inverse years squared). The vector $\mathbf{b}_{t}$ stacks $\sum_{s}\delta^{t-s}y_{s}$ and $\sum_{s}\delta^{t-s}g_{M,s}y_{s}$, and $c_{t}$ is the weighted second moment of the response. Together $(\mathbf{A}_{t}, \mathbf{b}_{t}, c_{t})$ are the __sufficient statistics__ of the weighted regression: any quantity EWLS reports at time $t$ is a function of these three running totals, and no raw observation needs to be kept.
___

## The Mechanical Recursion
Each moment is a weighted sum, and when time advances from $t-1$ to $t$ every old weight $\delta^{(t-1)-s}$ becomes $\delta\cdot\delta^{(t-1)-s} = \delta^{t-s}$, while the new observation $(\mathbf{x}_{t}, y_{t})$ enters with weight $\delta^{0} = 1$. Splitting each moment into its history and its today contribution gives, for the Gram matrix:
$$
\mathbf{A}_{t} = \sum_{s=1}^{t-1}\delta^{t-s}\,\mathbf{x}_{s}\mathbf{x}_{s}^{\top} + \mathbf{x}_{t}\mathbf{x}_{t}^{\top}
= \delta\sum_{s=1}^{t-1}\delta^{(t-1)-s}\,\mathbf{x}_{s}\mathbf{x}_{s}^{\top} + \mathbf{x}_{t}\mathbf{x}_{t}^{\top}
= \delta\,\mathbf{A}_{t-1} + \mathbf{x}_{t}\mathbf{x}_{t}^{\top}
$$
and identically for $\mathbf{b}_{t}$ and $c_{t}$. The recursion is therefore mechanical:
$$
\boxed{\;\mathbf{A}_{t} = \delta\,\mathbf{A}_{t-1} + \mathbf{x}_{t}\mathbf{x}_{t}^{\top},\qquad
\mathbf{b}_{t} = \delta\,\mathbf{b}_{t-1} + \mathbf{x}_{t}\,y_{t},\qquad
c_{t} = \delta\,c_{t-1} + y_{t}^{2}\quad\blacksquare\;}
$$
A two-by-two rank-one matrix add, a two-vector add, and a scalar add: the entire weighted past is compressed into $(\mathbf{A}_{t}, \mathbf{b}_{t}, c_{t})$ with no raw history retained. The recursion uses constant memory and constant work per step, and it runs once per asset; only $\delta$ and, below, the seed $\mathbf{A}_{0}$ are shared across assets.
___

## The Closed-Form Parameter Estimate
Solving the weighted normal equations $\mathbf{A}_{t}\boldsymbol{\theta} = \mathbf{b}_{t}$ gives the running SIM estimate:
$$
\boxed{\;\hat{\boldsymbol{\theta}}_{i,t} = \begin{pmatrix}\hat{\alpha}_{i,t}\\ \hat{\beta}_{i,t}\end{pmatrix} = \mathbf{A}_{t}^{-1}\,\mathbf{b}_{t}\quad\blacksquare\;}
$$
Because $\mathbf{A}_{t}$ is two-by-two, this is one direct solve per day (in code the system is solved, not inverted). Writing the entries as $\mathbf{A}_{t} = \begin{pmatrix}S_{w} & S_{wM}\\ S_{wM} & S_{wMM}\end{pmatrix}$ and $\mathbf{b}_{t} = \begin{pmatrix}S_{wy}\\ S_{wMy}\end{pmatrix}$, with $S_{w} = \sum_{s}\delta^{t-s}$, $S_{wM} = \sum_{s}\delta^{t-s}g_{M,s}$, $S_{wMM} = \sum_{s}\delta^{t-s}g_{M,s}^{2}$, $S_{wy} = \sum_{s}\delta^{t-s}y_{s}$, and $S_{wMy} = \sum_{s}\delta^{t-s}g_{M,s}y_{s}$ (all at time $t$; the subscript is suppressed), Cramer's rule gives:
$$
\hat{\beta}_{i,t} = \frac{S_{w}\,S_{wMy} - S_{wM}\,S_{wy}}{S_{w}\,S_{wMM} - S_{wM}^{2}},\qquad
\hat{\alpha}_{i,t} = \frac{S_{wy} - \hat{\beta}_{i,t}\,S_{wM}}{S_{w}}
$$
which are the OLS slope and intercept formulas of L6a with weighted moments in place of unweighted ones: EWLS is OLS on a re-weighted sample whose weights age exponentially in time. The denominator $S_{w}S_{wMM} - S_{wM}^{2}$ is $S_{w}^{2}$ times the weighted variance of the market growth rates, non-negative by the Cauchy–Schwarz inequality and zero only when the weighted market growth rates are all equal; the package skips the solve and keeps the previous estimate when it is numerically zero.
___

## The Residual Scale From the Same Moments
The residual scale falls out of the same sufficient statistics. Expanding the weighted residual sum of squares at any $\boldsymbol{\theta}$ gives:
$$
\sum_{s=1}^{t}\delta^{t-s}\left(y_{s} - \mathbf{x}_{s}^{\top}\boldsymbol{\theta}\right)^{2} = c_{t} - 2\,\boldsymbol{\theta}^{\top}\mathbf{b}_{t} + \boldsymbol{\theta}^{\top}\mathbf{A}_{t}\,\boldsymbol{\theta}
$$
At the optimum $\hat{\boldsymbol{\theta}}_{i,t}$ the normal equations give $\mathbf{A}_{t}\hat{\boldsymbol{\theta}}_{i,t} = \mathbf{b}_{t}$, so $\hat{\boldsymbol{\theta}}_{i,t}^{\top}\mathbf{A}_{t}\hat{\boldsymbol{\theta}}_{i,t} = \hat{\boldsymbol{\theta}}_{i,t}^{\top}\mathbf{b}_{t}$ and the last two terms partly cancel:
$$
\sum_{s=1}^{t}\delta^{t-s}\left(y_{s} - \mathbf{x}_{s}^{\top}\hat{\boldsymbol{\theta}}_{i,t}\right)^{2} = c_{t} - \hat{\boldsymbol{\theta}}_{i,t}^{\top}\mathbf{b}_{t}
$$
Dividing by the total weight $[\mathbf{A}_{t}]_{11} = S_{w}$ gives the residual scale estimate as a weighted root mean square residual:
$$
\boxed{\;\hat{\sigma}_{\varepsilon,i,t} = \sqrt{\frac{c_{t} - \hat{\boldsymbol{\theta}}_{i,t}^{\top}\,\mathbf{b}_{t}}{[\mathbf{A}_{t}]_{11}}}\quad\blacksquare\;}
$$
This is the exponentially weighted analogue of the L6a estimate $s^{2}_{g,\varepsilon} = \lVert\mathbf{r}\rVert_{2}^{2}/(N-2)$ __without__ the degrees-of-freedom correction (the denominator is the weight, not the weight less two); for the half-lives used in L7b (a month to a year) the difference is a few percent of $\hat{\sigma}$ at most, and the residual scale is reported, not used for inference. Its units are inverse years, like the growth rates.
___

## Prior Seeding: Why the Seeded State Returns the Prior
A recursion has to start somewhere, and $\mathbf{A}_{0} = \mathbf{0}$ leaves the first days unidentified. L7b seeds it with the archive estimate, worth a stated number of pseudo-observations. Let $\boldsymbol{\theta}_{i,0} = (\alpha_{i,0}, \beta_{i,0})^{\top}$ and $\sigma_{\varepsilon,i,0}$ be the prior (the archive's $\hat{\alpha}_{i}$, $\hat{\beta}_{i}$, $s_{g,\varepsilon,i}$), let $N_{0} > 0$ be the __prior weight__ in pseudo-observations, and let $\mathbf{M} = \mathbb{E}[\mathbf{x}\mathbf{x}^{\top}]$ be the second-moment matrix of the regressor on the calibration sample, which for $\mathbf{x} = (1, g_{M})^{\top}$ is built from the training-period market mean $g^{\prime}_{M}$ and variance $s^{2}_{g,M}$ of the archive:
$$
\mathbf{M} = \mathbb{E}\left[\mathbf{x}\mathbf{x}^{\top}\right] = \begin{pmatrix}1 & g^{\prime}_{M}\\ g^{\prime}_{M} & s^{2}_{g,M} + g^{\prime\,2}_{M}\end{pmatrix}
$$
The seed is the state that $N_{0}$ typical calibration days consistent with the prior would have produced:
$$
\boxed{\;\mathbf{A}_{0} = N_{0}\,\mathbf{M},\qquad
\mathbf{b}_{0} = \mathbf{A}_{0}\,\boldsymbol{\theta}_{i,0},\qquad
c_{0} = \boldsymbol{\theta}_{i,0}^{\top}\mathbf{b}_{0} + N_{0}\,\sigma_{\varepsilon,i,0}^{2}\;}
$$
> __Proposition (the seed returns the prior):__ Before any observation is processed, the recursion's estimate is $\hat{\boldsymbol{\theta}}_{i,0} = \boldsymbol{\theta}_{i,0}$ and its residual scale is $\hat{\sigma}_{\varepsilon,i,0} = \sigma_{\varepsilon,i,0}$, exactly.
>
> __Proof.__ $\mathbf{M}$ is positive definite whenever $s^{2}_{g,M} > 0$ (its determinant is $s^{2}_{g,M}$), so $\mathbf{A}_{0}$ is invertible and $\mathbf{A}_{0}^{-1}\mathbf{b}_{0} = \mathbf{A}_{0}^{-1}\mathbf{A}_{0}\boldsymbol{\theta}_{i,0} = \boldsymbol{\theta}_{i,0}$. For the scale, $c_{0} - \hat{\boldsymbol{\theta}}_{i,0}^{\top}\mathbf{b}_{0} = \boldsymbol{\theta}_{i,0}^{\top}\mathbf{b}_{0} + N_{0}\sigma_{\varepsilon,i,0}^{2} - \boldsymbol{\theta}_{i,0}^{\top}\mathbf{b}_{0} = N_{0}\sigma_{\varepsilon,i,0}^{2}$, and $[\mathbf{A}_{0}]_{11} = N_{0}$, so the ratio is $\sigma_{\varepsilon,i,0}^{2}$. $\blacksquare$

The recursion therefore launches at the prior and moves only when real data arrive. Two consequences of the seed matter for how it moves. Every residual receives its age weight, but the information a day carries about the slope scales with the square of its centered market growth (what the day adds to the slope direction of $\mathbf{A}_{t}$), so a real day with a large market move carries more slope information than one pseudo-observation and the prior lets go of the slope faster than $N_{0}$ days would suggest on days like that; and if the calibration growth rates were smoother than the ones the recursion updates on (the L7b archive was estimated on volume-weighted average prices, the recursion updates on closes), that effect is systematic. One more small point of exactness: $\mathbf{M}$ is the population second moment, so if $s^{2}_{g,M}$ is the corrected sample variance the exact empirical second moment is $(N-1)s^{2}_{g,M}/N + g^{\prime\,2}_{M}$, a difference of order $1/N$ that is immaterial for the course sample.
___

## Prior Seeding: The Decaying Prior-Centered Quadratic
What objective does the seeded recursion minimize? Apply the recursion $t$ times from the seed. Because every step multiplies the existing moments by $\delta$ before adding the new observation, the seed's contribution after $t$ steps is $\delta^{t}$ times the seed:
$$
\mathbf{A}_{t} = \delta^{t}\mathbf{A}_{0} + \sum_{s=1}^{t}\delta^{t-s}\mathbf{x}_{s}\mathbf{x}_{s}^{\top},\qquad
\mathbf{b}_{t} = \delta^{t}\mathbf{A}_{0}\boldsymbol{\theta}_{i,0} + \sum_{s=1}^{t}\delta^{t-s}\mathbf{x}_{s}y_{s}
$$
so the prior's weight after $t$ updates is $\delta^{t}N_{0}$: it decays like the data, halving every $t_{1/2}$ days.

> __Proposition (the seeded objective):__ For $t\ge 0$, the seeded recursion's estimate $\hat{\boldsymbol{\theta}}_{i,t} = \mathbf{A}_{t}^{-1}\mathbf{b}_{t}$ is the unique minimizer of the data loss plus a decaying prior-centered quadratic:
> $$
J_{t}(\boldsymbol{\theta}) = \underbrace{\sum_{s=1}^{t}\delta^{t-s}\left(y_{s} - \mathbf{x}_{s}^{\top}\boldsymbol{\theta}\right)^{2}}_{\text{data loss }L_{t}(\boldsymbol{\theta})} + \underbrace{\delta^{t}N_{0}\left(\boldsymbol{\theta} - \boldsymbol{\theta}_{i,0}\right)^{\top}\mathbf{M}\left(\boldsymbol{\theta} - \boldsymbol{\theta}_{i,0}\right)}_{\text{decaying prior}}
$$
>
> __Proof.__ $J_{t}$ is a strictly convex quadratic (its Hessian is $2\mathbf{A}_{t}$, positive definite because $\delta^{t}N_{0}\mathbf{M}$ is), so its stationary point is its unique minimizer. Its gradient is $-2\sum_{s\le t}\delta^{t-s}\mathbf{x}_{s}(y_{s} - \mathbf{x}_{s}^{\top}\boldsymbol{\theta}) + 2\delta^{t}N_{0}\mathbf{M}(\boldsymbol{\theta} - \boldsymbol{\theta}_{i,0})$; setting it to zero and collecting terms gives $\left(\sum_{s\le t}\delta^{t-s}\mathbf{x}_{s}\mathbf{x}_{s}^{\top} + \delta^{t}N_{0}\mathbf{M}\right)\boldsymbol{\theta} = \sum_{s\le t}\delta^{t-s}\mathbf{x}_{s}y_{s} + \delta^{t}N_{0}\mathbf{M}\boldsymbol{\theta}_{i,0}$, which is $\mathbf{A}_{t}\boldsymbol{\theta} = \mathbf{b}_{t}$ with the seeded moments above. $\blacksquare$

So the lecture's theorem, which minimizes the data-only loss $L_{t}$, and the seeded recursion, which minimizes $J_{t}$, are two objectives; their minimizers approach one another as $\delta^{t}N_{0}$ becomes small against the accumulated data weight (provided the data Gram matrix stays well conditioned), and before that the estimate is a weighted compromise between the prior and the data. The prior is a regularizer toward the archive, with the calibration sample's geometry $\mathbf{M}$ and a weight that decays; it is not the ridge estimator of L6a's optional material, which shrinks toward zero with a fixed penalty (both modify the normal equations, and that is where the resemblance ends). Two knobs and only two: the half-life $t_{1/2}$ sets how fast both the prior and the data fade, and the prior weight $N_{0}$ sets how much the prior counts on the first day. A short half-life (a month) tracks quickly and inherits daily noise, above all in the intercept; a long one (a year) is smooth and lags; the L7b examples choose between them by walk-forward prediction inside the training period.
___

## Summary
The EWLS recursion compresses the entire weighted history of asset $i$'s SIM regression into three running moments $(\mathbf{A}_{t}, \mathbf{b}_{t}, c_{t})$, updated each trading day by one decayed-plus-rank-one step; the parameter estimate comes from a two-by-two solve, the residual scale falls out of the same moments through a cancellation at the optimum, and the prior enters as $N_{0}$ pseudo-observations that decay like the data.

> __Key Takeaways:__
>
> * __Three running moments are sufficient:__ The weighted normal equations show that the weighted Gram matrix, cross-moment vector, and response second moment are the only quantities the estimator needs from the weighted history, so no raw observation is retained and the recursion runs in constant memory and constant work per day.
> * __The recursion is one decayed-plus-rank-one step, and the closed forms follow:__ Advancing time multiplies every moment by the decay factor and adds today's contribution, the intercept and beta are the OLS formulas with weighted moments, and the residual scale is the weighted root mean square residual without a degrees-of-freedom correction.
> * __The seed returns the prior exactly and then fades:__ Seeding the moments with a stated number of pseudo-observations built from the calibration second-moment matrix returns the prior before any data, and the seeded recursion minimizes the data loss plus a prior-centered quadratic whose weight is the prior weight times the decay factor raised to the number of updates, so the half-life and the prior weight are the only knobs and the estimate is a weighted compromise between the archive and the recent past.

The L7b lecture runs this recursion on close-based growth rates for thirteen firms, chooses the half-life inside the training period, and replays the L7a engine with the resulting parameter paths.
___